# Step 2: Baselines, including team 3's hybrid NCF


**Pipeline.** `step1_data_preprocess.ipynb` → `step2_baselines.ipynb` (this notebook) → `step3_lightgbm.ipynb` (base model) → `step3_lambdarank.ipynb` (final model) and `step3_attention.ipynb` (extension)

This notebook scores every baseline on the **same users and the same full slates** that our model is scored on in step 3:

| Baseline | Score | What it tests |
|---|---|---|
| Random | random number | the floor. Expected Hit@10 ≈ 22.7% |
| Popularity | actions on the product across train users' histories | non-personalised, the fallback for users with little history |
| Repeat → popularity | products in the user's own history first, then the rest by popularity | the same-product signal (13.3× lift, interim report Table 6). The strongest non-learned rule, so the real bar for our model |
| Query-only | share of query terms in the product name | whether query matching alone helps. Every candidate was retrieved for the query, so we expect it just above random |
| MF (CF) | user · product factors from step 1 (D13) | the conventional collaborative-filtering approach (the NCF/NMF family) |
| **Team 3 hybrid NCF** | their LSTM + attention model, retrained on our split (D15, D16) | **our prior baseline**: the source code we set out to improve |

The second half of the notebook is the **evaluation-protocol check**: team 3 reported Hit@10 0.978 and NDCG@10 0.568, measured on lists with 4 random negatives per positive. We score their model on those lists and on full slates, so the report can say how much of that number came from the protocol.

## How to run in Colab
1. Step 1's files are read from the shared BT4222_artifacts folder by its id; you do not need to run step 1 first.
2. **Runtime → Change runtime type → T4 GPU.** Team 3's model is a PyTorch LSTM; on a T4 one epoch over the full train set takes a few minutes. It also runs on CPU, much more slowly (use step 1's `MAX_USERS` for a quick check). Team 3's training takes ~70 min on a T4 (10 epochs).
3. Runtime → Run all. Result cells are marked **RESULT**. Keep all outputs when you save.
4. The last cell uploads baseline_scores.parquet to the shared folder only if UPLOAD_ARTIFACTS = True.

| # | Decision | Choice |
|---|---|---|
| D10 | Evaluation | Full slates, Hit@10, NDCG@10, MRR and P@1, on train (D12), validation and test users. NDCG gain is set by `NDCG_GAIN`: `"linear"` (click 1, cart 2, purchase 3, as in the interim report and `src/metrics.py`) or `"exp"` (2^label − 1). **We use linear; step 3 uses the same.** Ties are broken at random |
| D15 | Team 3 port | Their architecture, loss, optimiser and schedule, on our split and our full history (details in step 1, Section 8) |
| D16 | Team 3 scoring | Reported both as they ranked (highest class probability) and by P(engaged) = 1 − P(no engagement) |

## 1. Setup and load the step 1 artifacts

In [ ]:
import os, sys, time, gc, json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

IN_COLAB = "google.colab" in sys.modules

# ---- config: change these, nothing else ----
ART_DIR = os.environ.get("JD_ART_DIR", "/content/artifacts")       # step 1 output (downloaded from Drive)
OUT_DIR = os.environ.get("JD_OUT_DIR", "/content/results")          # metrics and plots are saved here
DRIVE_ARTIFACT_FOLDER = "BT4222_artifacts"                  # shared folder the group reads (steps 2 and 3)
ARTIFACT_FOLDER_ID = "1Yb4AGglIJV4Pdd8Sx6NozG8Klh6uweqH"   # its Drive id, same for everyone
UPLOAD_ARTIFACTS = False                                    # True only to replace the shared files (needs edit access)
T3_EPOCHS = int(os.environ.get("JD_T3_EPOCHS", 10))   # team 3 trained for 10 epochs
T3_BATCH = 32                  # users per batch, as in team 3
T3_LR = 1e-3                   # Adam, weight decay 1e-5, as in team 3
NDCG_GAIN = "linear"           # "linear": gain = label (interim report, src/metrics.py); "exp": 2^label - 1. Same in step 3
# --------------------------------------------
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def _drive():
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    return build("drive", "v3")

def _folder_id(svc, create=False):
    return ARTIFACT_FOLDER_ID   # shared BT4222_artifacts, found by id (not by name in your own My Drive)

def drive_upload(names):
    # copy ART_DIR/<name> into My Drive/DRIVE_ARTIFACT_FOLDER (created if missing); re-running overwrites
    from googleapiclient.http import MediaFileUpload
    svc = _drive(); fid = _folder_id(svc, create=True)
    for f in names:
        t0 = time.time()
        old = svc.files().list(q=f"'{fid}' in parents and name='{f}' and trashed=false", fields="files(id)").execute()["files"]
        media = MediaFileUpload(f"{ART_DIR}/{f}", resumable=True, chunksize=100 * 1024 * 1024)
        if old:
            svc.files().update(fileId=old[0]["id"], media_body=media).execute()
        else:
            svc.files().create(body={"name": f, "parents": [fid]}, media_body=media).execute()
        print(f"uploaded {f} to My Drive/{DRIVE_ARTIFACT_FOLDER}  {time.time() - t0:.0f}s")

def drive_download(names):
    import io
    from googleapiclient.http import MediaIoBaseDownload
    names = [f for f in names if not os.path.exists(f"{ART_DIR}/{f}")]
    if not names:
        return
    svc = _drive(); fid = _folder_id(svc, create=False); os.makedirs(ART_DIR, exist_ok=True)
    for f in names:
        found = svc.files().list(q=f"'{fid}' in parents and name='{f}' and trashed=false",
                                 fields="files(id, size)").execute(num_retries=8)["files"]
        if not found:
            raise FileNotFoundError(f"{f} is not in My Drive/{DRIVE_ARTIFACT_FOLDER}. Run the step that makes it first.")
        tmp = f"{ART_DIR}/{f}.part"
        with io.FileIO(tmp, "wb") as fh:
            dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=found[0]["id"]), chunksize=100 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk(num_retries=8)
        assert os.path.getsize(tmp) == int(found[0]["size"]), f"incomplete download: {f}"
        os.replace(tmp, f"{ART_DIR}/{f}")
        print("downloaded", f)

STEP1_FILES = ["features.parquet", "users.parquet", "config.json", "team3_meta.parquet", "team3_seq.npz"]
STEP2_FILES = ["baseline_scores.parquet"]

if IN_COLAB and DRIVE_ARTIFACT_FOLDER:
    drive_download(STEP1_FILES)
for f in STEP1_FILES:
    if not os.path.exists(f"{ART_DIR}/{f}"):
        raise FileNotFoundError(f"{ART_DIR}/{f} not found. Run step 1 to the end first (its last cell copies the files to your Drive).")

config = json.load(open(f"{ART_DIR}/config.json"))
SEED, T3_META, T3_SIZES = config["SEED"], config["T3_META"], config["T3_SIZES"]
T3_COLS = ["t3_wid", "t3_brand_id", "t3_shop_id", "t3_cate_id_1", "t3_cate_id_2", "t3_cate_id_3", "t3_cate_id_4"]
cols = ["user", "wid", "label", "split", "fit", "train_eval", "t3_list", "pop_w0", "pop_w1", "pop_w2", "pop_impr_train",
        "name_overlap", "item_days_since", "score_mf"] + T3_COLS
K = pd.read_parquet(f"{ART_DIR}/features.parquet", columns=cols)
U = pd.read_parquet(f"{ART_DIR}/users.parquet")
M = pd.read_parquet(f"{ART_DIR}/team3_meta.parquet").set_index("user").sort_index()
S = np.load(f"{ART_DIR}/team3_seq.npz")
SEQ = {k: S[k] for k in ["item", "type", "time", "length"]}
SPLITS = {0: "train", 1: "val", 2: "test"}
torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", device, "| pandas", pd.__version__)
print(f"{len(K):,} rows, {len(U):,} users, team 3 vocabularies: {T3_SIZES}")

assert config["MAX_USERS"] == 0 and len(U) == 171728 and len(K) == 6690773, "not the full step 1 output"

downloaded features.parquet
downloaded users.parquet
downloaded config.json
downloaded team3_meta.parquet
downloaded team3_seq.npz
torch 2.11.0+cu130 | device cuda | pandas 2.2.3
6,690,773 rows, 171,728 users, team 3 vocabularies: {'item': 1000001, 'type': 5, 'time': 11, 'brand_id': 118703, 'shop_id': 190634, 'cate_id_1': 68, 'cate_id_2': 596, 'cate_id_3': 5062, 'cate_id_4': 6249}


### Sample splits used for training and evaluation

Users were split 80/10/10 in step 1, stratified by history length (D3). Every metric below is computed per user on **full slates**: the 10,000 train users that kept their full slate (training performance, D12), and all validation and test users. Validation is used for early stopping and model choice; test is reported once.

Team 3's model is trained the way they trained it, on **their lists** from train users: every positive plus 4 random negatives per positive (`t3_list`). One limit: step 1 keeps only 20 negatives per train slate (D8), so a train user with more than 5 positives gets 20 negatives rather than 4 per positive. That affects few slates (the count is printed below).

In [ ]:
tr_eval = (K.split.values == 0) & K.train_eval.values
va, te = K.split.values == 1, K.split.values == 2
EVAL_SETS = {"train": tr_eval, "val": va, "test": te}
t3_fit = (K.split.values == 0) & K.t3_list.values
sizes = pd.DataFrame({
    "users": [K.user[t3_fit].nunique()] + [K.user[m].nunique() for m in EVAL_SETS.values()],
    "rows": [t3_fit.sum()] + [m.sum() for m in EVAL_SETS.values()],
    "engaged share": [(K.label[t3_fit] > 0).mean()] + [(K.label[m] > 0).mean() for m in EVAL_SETS.values()]},
    index=["team 3 training lists (train users)", "train (full slates, D12)", "val (full slates)", "test (full slates)"])
display(sizes.round(4))
tr_users = U[U.split == 0]
print(f"train slates with more than 5 positives (negatives capped at 20): {(tr_users.n_pos > 5).mean():.1%}")

,users,rows,engaged share
team 3 training lists (train users),137380,1213778,0.2113
"train (full slates, D12)",10000,893691,0.0207
val (full slates),17172,1538177,0.0210
test (full slates),17176,1535067,0.0209


train slates with more than 5 positives (negatives capped at 20): 3.6%


## 2. Evaluation helper

One definition of every metric, shared with step 3 (`src/metrics.py`, with an identical copy for Colab). A slate is ranked by the score, **ties are broken at random**, and slates with no engaged product are skipped. Confidence intervals are bootstrapped over users.

In [ ]:
# Shared metric definitions (src/metrics.py). On Colab without the repo, an identical copy is defined here.
try:
    from src.metrics import MetricAccumulator, bootstrap_ci
except ImportError:
    def _rank(scores, rng):
        # descending score, ties broken uniformly at random (never by stored order: the raw file stores engaged items first)
        return np.lexsort((rng.random(len(scores)), -np.asarray(scores, float)))
    def _metrics(ranked, k, gain):
        g = ranked.astype(float) if gain == "linear" else np.power(2.0, ranked) - 1.0
        rel = np.flatnonzero(ranked > 0)
        disc = 1.0 / np.log2(np.arange(2, k + 2))
        dcg = (g[:k] * disc[: len(g[:k])]).sum()
        ideal = np.sort(g)[::-1][:k]
        idcg = (ideal * disc[: len(ideal)]).sum()
        return {f"hit@{k}": float((ranked[:k] > 0).any()), f"ndcg@{k}": float(dcg / idcg) if idcg > 0 else 0.0,
                "mrr": float(1.0 / (rel[0] + 1)) if len(rel) else 0.0, "p@1": float(ranked[0] > 0)}
    class MetricAccumulator:
        def __init__(self, k=10, gain="linear", seed=42):
            self.k, self.gain, self.rng, self.rows = k, gain, np.random.default_rng(seed), []
        def add(self, scores, labels):
            labels = np.asarray(labels)
            if not (labels > 0).any():
                return None
            m = _metrics(labels[_rank(scores, self.rng)], self.k, self.gain); self.rows.append(m); return m
        def per_slate(self):
            return {key: np.array([r[key] for r in self.rows]) for key in self.rows[0]} if self.rows else {}
    def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=0):
        values = np.asarray(list(values), float); rng = np.random.default_rng(seed)
        means = values[rng.integers(0, len(values), size=(n_boot, len(values)))].mean(axis=1)
        return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

METRICS = ["hit@10", "ndcg@10", "mrr", "p@1"]

def evaluate(df, score):
    """Rank each slate in df (rows grouped by user) by score; one row of metrics per slate."""
    users = df.user.values
    assert (np.diff(users) >= 0).all(), "rows must be grouped by user"
    starts = np.r_[0, np.flatnonzero(np.diff(users)) + 1]; ends = np.r_[starts[1:], len(users)]
    s, y = np.asarray(score, float), df.label.values
    acc, kept = MetricAccumulator(gain=NDCG_GAIN, seed=SEED), []
    for a, b in zip(starts, ends):
        if acc.add(s[a:b], y[a:b]) is not None:
            kept.append(users[a])
    out = pd.DataFrame(acc.per_slate()); out.insert(0, "user", kept)
    return out

def summarise(per, name):
    row = {"model": name, "slates": len(per)}
    row.update({m: per[m].mean() for m in METRICS})
    for m in ["hit@10", "ndcg@10"]:
        lo, hi = bootstrap_ci(per[m]); row[f"{m} 95% CI"] = f"{lo:.3f} to {hi:.3f}"
    return row

FMT = {m: "{:.4f}" for m in METRICS} | {"slates": "{:,}"}
BASELINE_NAMES = {"score_random": "Random", "score_query": "Query only", "score_pop": "Popularity",
                  "score_repeat_pop": "Repeat → popularity", "score_mf": "MF (truncated SVD)",
                  "score_t3_maxprob": "Team 3 NCF (as published: max class prob.)",
                  "score_t3_pengaged": "Team 3 NCF (fixed: P(engaged))"}

# sanity check: a constant score must perform like random (~0.227). Near 1.0 would mean the stored order is leaking
V0 = K[va].reset_index(drop=True)
print(f"all-zero score, validation Hit@10 = {evaluate(V0, np.zeros(len(V0)))['hit@10'].mean():.4f} (random ≈ 0.227)")

all-zero score, validation Hit@10 = 0.2246 (random ≈ 0.227)


## 3. Non-learned baselines and MF

Scored for every evaluation row (train D12, validation, test). None of these is fitted on slate labels, so nothing here is tuned on validation or test.

Repeat → popularity uses each user's full history (step 1, D14: no cap).

**Note:** step 1's F7 fix changed popularity for 6,720 train rows after this notebook ran. Validation and test scores are unaffected; `step3_lightgbm` (Section 1) recomputes the train scores.

In [ ]:
EV = K[tr_eval | va | te].reset_index(drop=True)          # evaluation rows, still grouped by user
rng = np.random.default_rng(SEED)
pop = (EV.pop_w0 + EV.pop_w1 + EV.pop_w2).to_numpy(float)
SC = EV[["user", "wid", "split", "label"]].copy()
SC["score_random"] = rng.random(len(EV))
SC["score_query"] = EV.name_overlap.fillna(0).to_numpy(float)
SC["score_pop"] = pop
SC["score_repeat_pop"] = EV.item_days_since.notna().to_numpy() * 1e9 + pop
SC["score_mf"] = EV.score_mf.to_numpy(float)
SC["train_eval"] = EV.train_eval.values
print(f"scored {len(SC):,} rows in {SC.user.nunique():,} slates")

scored 3,966,935 rows in 44,348 slates


**RESULT: non-learned baselines and MF** (validation users, full slates)

In [ ]:
def results_for(split_code, cols):
    part = SC[SC.split.values == split_code].reset_index(drop=True)
    return pd.DataFrame([summarise(evaluate(part, part[c]), BASELINE_NAMES.get(c, c)) for c in cols]).set_index("model")

SIMPLE = ["score_random", "score_query", "score_pop", "score_repeat_pop", "score_mf"]
res_val = results_for(1, SIMPLE)
display(res_val.style.format(FMT))
res_val.to_csv(f"{OUT_DIR}/step2_val_simple_baselines.csv")

# how often popularity and MF have nothing to go on (score 0 = tied with every other unseen product)
V = SC[SC.split.values == 1]
display(pd.DataFrame({
    "candidate rows with score 0": {BASELINE_NAMES[c]: (V[c].values == 0).mean() for c in ["score_pop", "score_mf"]},
    "ENGAGED rows with score 0": {BASELINE_NAMES[c]: (V[c].values[V.label.values > 0] == 0).mean() for c in ["score_pop", "score_mf"]},
}).style.format("{:.1%}"))

,slates,hit@10,ndcg@10,mrr,p@1,hit@10 95% CI,ndcg@10 95% CI
model,,,,,,,
Random,"17,172",0.2240,0.0826,0.1008,0.0302,0.218 to 0.230,0.080 to 0.085
Query only,"17,172",0.2349,0.0912,0.1085,0.0368,0.228 to 0.241,0.088 to 0.094
Popularity,"17,172",0.6437,0.3448,0.3388,0.1972,0.637 to 0.651,0.340 to 0.350
Repeat → popularity,"17,172",0.6813,0.3856,0.3869,0.2463,0.675 to 0.689,0.381 to 0.391
MF (truncated SVD),"17,172",0.6029,0.3196,0.3181,0.1837,0.596 to 0.610,0.315 to 0.324


,candidate rows with score 0,ENGAGED rows with score 0
Popularity,59.0%,30.3%
MF (truncated SVD),56.7%,26.7%


## 4. Team 3 hybrid NCF (prior baseline)

Team 3's model (`3.1_Hybrid_NCF_with_LSTM.ipynb`), rebuilt on our data and split (D15):
- **User history encoder:** each of the last 200 actions is the sum of a product-ID, action-type and time-gap-bin embedding. A 2-layer LSTM reads the sequence and attention pools it into one user vector. Users with no history get a learned "empty" vector.
- **Candidate encoder:** sum of product-ID, brand, shop and four category embeddings. We look the attributes up directly (step 1, fix 2 in D15).
- **Meta features:** their 15 user features, missing values set to 0, standardised (scaler fitted on train users), then one linear layer.
- **Classifier:** the three vectors are concatenated and an MLP (128 → 64 → 4, ReLU, batch norm, dropout 0.3) predicts none / click / cart / purchase.
- **Training, as theirs:** class-weighted cross-entropy (weights ∝ 1 / class count), Adam (lr 1e-3, weight decay 1e-5), gradient clipping at 1.0, learning rate halved when validation loss stalls for 2 epochs, 32 users per batch, `T3_EPOCHS` epochs, best epoch chosen by validation loss. Trained on **their lists** from train users (`t3_fit`: every positive plus 4 negatives per positive); validation loss on validation users' lists.
- **Scoring (D16):** `score_t3_maxprob` ranks by the highest class probability (their rule) and `score_t3_pengaged` by P(engaged) = 1 − P(none).

In [ ]:
class Attention(nn.Module):
    """Team 3's attention pooling: one score per time step, softmax over real (non-padded) steps."""
    def __init__(self, emb_dim):
        super().__init__()
        self.attn = nn.Linear(emb_dim, 1)
    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1).masked_fill(mask == 0, float("-inf"))
        return (x * torch.softmax(scores, dim=1).unsqueeze(-1)).sum(dim=1)

T3_CAND_KEYS = ["item", "brand_id", "shop_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4"]

class T3CandidateEncoder(nn.Module):
    """Sum of product-ID, brand, shop and 4 category embeddings (team 3), attributes passed in directly."""
    def __init__(self, sizes, emb_dim=64):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(sizes[k], emb_dim, padding_idx=0) for k in T3_CAND_KEYS])
    def forward(self, cand):                       # cand: [batch, candidates, 7]
        return sum(emb(cand[..., i]) for i, emb in enumerate(self.embs))

class SequentialHybridNCF(nn.Module):
    """Team 3's SequentialHybridNCF: LSTM + attention user encoder, candidate encoder, meta features, MLP to 4 classes."""
    def __init__(self, sizes, num_meta, emb_dim=64, hidden_dim=128, num_classes=4):
        super().__init__()
        self.item_emb = nn.Embedding(sizes["item"], emb_dim, padding_idx=0)
        self.type_emb = nn.Embedding(sizes["type"], emb_dim, padding_idx=0)
        self.time_emb = nn.Embedding(sizes["time"], emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=emb_dim, batch_first=True, dropout=0.3, num_layers=2)
        self.attention = Attention(emb_dim)
        self.empty_user_emb = nn.Parameter(torch.zeros(1, emb_dim))
        self.candidate_encoder = T3CandidateEncoder(sizes, emb_dim)
        self.meta_linear = nn.Linear(num_meta, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 3, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.BatchNorm1d(hidden_dim // 2), nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, num_classes))

    def encode_user_history(self, wid_seq, type_seq, time_seq, lengths):
        # real actions = positions before each user's length (team 3 used wid_seq != 0, which gives NaN when every
        # action of a user is outside the 1M-product vocabulary and so has id 0)
        mask = torch.arange(wid_seq.size(1), device=wid_seq.device).unsqueeze(0) < lengths.unsqueeze(1)
        x = self.item_emb(wid_seq) + self.type_emb(type_seq) + self.time_emb(time_seq)
        out_emb = self.empty_user_emb.repeat(x.size(0), 1)
        has = lengths > 0
        if has.any():
            packed = pack_padded_sequence(x[has], lengths[has].cpu(), batch_first=True, enforce_sorted=False)
            out, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True, total_length=x.size(1))
            out_emb[has] = self.attention(out, mask[has])
        return out_emb

    def forward(self, wid_seq, type_seq, time_seq, lengths, cand, meta):
        B, C = cand.shape[:2]
        user = self.encode_user_history(wid_seq, type_seq, time_seq, lengths).unsqueeze(1).expand(-1, C, -1)
        meta_e = self.meta_linear(meta).unsqueeze(1).expand(-1, C, -1)
        fusion = torch.cat([user, self.candidate_encoder(cand), meta_e], dim=-1)
        return self.mlp(fusion.reshape(-1, fusion.size(-1))).reshape(B, C, -1)

In [ ]:
# ---- inputs: meta features, history sequences, candidate attribute indices ----
from sklearn.preprocessing import StandardScaler
meta = M[T3_META].fillna(0).to_numpy(np.float32)                          # team 3: fillna(0), then StandardScaler
tr_user = (U.set_index("user").split.reindex(M.index).values == 0)
META_T = torch.tensor(StandardScaler().fit(meta[tr_user]).transform(meta), dtype=torch.float32)   # scaler fitted on train users
SEQ_T = {k: torch.from_numpy(SEQ[k].astype(np.int64)) for k in ["item", "type", "time"]}
LEN_T = torch.from_numpy(SEQ["length"].astype(np.int64))
CAND_ALL = K[T3_COLS].to_numpy(np.int64)                                   # t3_wid, brand, shop, cate 1-4 per row
LAB_ALL = K.label.to_numpy(np.int64)
T3_SZ = dict(T3_SIZES)

def groups(mask):
    """Users and row positions for the rows in mask (rows are grouped by user)."""
    idx = np.flatnonzero(mask)
    users, start = np.unique(K.user.values[idx], return_index=True)
    return users, np.split(idx, start[1:])

def batches(users, rows, batch_users=T3_BATCH, shuffle=False, seed=SEED):
    order = np.random.default_rng(seed).permutation(len(users)) if shuffle else np.arange(len(users))
    for s in range(0, len(order), batch_users):
        sel = order[s: s + batch_users]; u = torch.as_tensor(users[sel], dtype=torch.long)
        L = int(LEN_T[u].max().clamp(min=1))
        C = max(len(rows[i]) for i in sel)
        cand = torch.zeros((len(sel), C, len(T3_COLS)), dtype=torch.long); lab = torch.full((len(sel), C), -1, dtype=torch.long)
        for j, i in enumerate(sel):
            r = rows[i]; cand[j, :len(r)] = torch.from_numpy(CAND_ALL[r]); lab[j, :len(r)] = torch.from_numpy(LAB_ALL[r])
        yield (sel, SEQ_T["item"][u, :L], SEQ_T["type"][u, :L], SEQ_T["time"][u, :L], LEN_T[u], cand, META_T[u], lab)

fit_users, fit_rows = groups(t3_fit)
val_users, val_rows = groups(va & K.t3_list.values)                       # validation loss on their lists, as they did
print(f"team 3 training: {len(fit_users):,} users, {t3_fit.sum():,} rows | validation lists: {len(val_users):,} users")

team 3 training: 137,380 users, 1,213,778 rows | validation lists: 17,172 users


In [ ]:
# ---- train (their loss, optimiser and schedule) ----
counts = np.bincount(LAB_ALL[t3_fit], minlength=4).astype(np.float64)
class_w = torch.tensor((1 / counts) / (1 / counts).sum(), dtype=torch.float32, device=device)   # weights ∝ 1 / class count
criterion = nn.CrossEntropyLoss(weight=class_w)
t3_model = SequentialHybridNCF(T3_SZ, len(T3_META)).to(device)
optimizer = torch.optim.Adam(t3_model.parameters(), lr=T3_LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

def run_epoch(users, rows, train):
    t3_model.train(train); total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for _, ws, ts, tms, ln, cand, mt, lab in batches(users, rows, shuffle=train, seed=SEED + epoch):
            ws, ts, tms, ln, cand, mt, lab = [t.to(device) for t in (ws, ts, tms, ln, cand, mt, lab)]
            logits = t3_model(ws, ts, tms, ln, cand, mt)
            m = lab != -1
            loss = criterion(logits[m], lab[m])
            if train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(t3_model.parameters(), max_norm=1.0); optimizer.step()
            total += loss.item(); n += 1
    return total / max(n, 1)

best_loss, best_state, t0 = float("inf"), None, time.time()
for epoch in range(T3_EPOCHS):
    tr_loss = run_epoch(fit_users, fit_rows, train=True)
    assert np.isfinite(tr_loss), "training loss is NaN: check the inputs"
    va_loss = run_epoch(val_users, val_rows, train=False)
    scheduler.step(va_loss)
    if va_loss < best_loss:
        best_loss, best_epoch = va_loss, epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in t3_model.state_dict().items()}
        torch.save({"state": best_state, "epoch": best_epoch, "val_loss": best_loss}, f"{ART_DIR}/t3_best.pt")
        if IN_COLAB and UPLOAD_ARTIFACTS:
            try:
                drive_upload(["t3_best.pt"])
            except Exception as e:
                print("checkpoint upload failed:", e)
    print(f"epoch {epoch + 1}/{T3_EPOCHS}: train loss {tr_loss:.4f} | val loss {va_loss:.4f} | lr {optimizer.param_groups[0]['lr']:.6f} | {time.time() - t0:.0f}s")
t3_model.load_state_dict(best_state)
print(f"best epoch {best_epoch} (val loss {best_loss:.4f})")

uploaded t3_best.pt to My Drive/BT4222_artifacts  19s
epoch 1/10: train loss 1.3122 | val loss 1.2651 | lr 0.001000 | 440s


uploaded t3_best.pt to My Drive/BT4222_artifacts  21s
epoch 2/10: train loss 1.2616 | val loss 1.2499 | lr 0.001000 | 881s


uploaded t3_best.pt to My Drive/BT4222_artifacts  22s
epoch 3/10: train loss 1.2367 | val loss 1.2410 | lr 0.001000 | 1319s


uploaded t3_best.pt to My Drive/BT4222_artifacts  20s
epoch 4/10: train loss 1.2165 | val loss 1.2389 | lr 0.001000 | 1748s
epoch 5/10: train loss 1.1986 | val loss 1.2392 | lr 0.001000 | 2139s
epoch 6/10: train loss 1.1810 | val loss 1.2427 | lr 0.001000 | 2531s
epoch 7/10: train loss 1.1630 | val loss 1.2488 | lr 0.000500 | 2924s
epoch 8/10: train loss 1.0897 | val loss 1.2909 | lr 0.000500 | 3316s
epoch 9/10: train loss 1.0555 | val loss 1.3201 | lr 0.000500 | 3707s
epoch 10/10: train loss 1.0308 | val loss 1.3522 | lr 0.000250 | 4099s
best epoch 4 (val loss 1.2389)


In [ ]:
# ---- score every evaluation row (full slates) ----
ev_mask = tr_eval | va | te
ev_users, ev_rows = groups(ev_mask)
ev_pos = np.full(len(K), -1, np.int64); ev_pos[np.flatnonzero(ev_mask)] = np.arange(ev_mask.sum())   # K row -> SC row
maxprob = np.zeros(len(SC), np.float32); pengaged = np.zeros(len(SC), np.float32)
t3_model.eval()
with torch.no_grad():
    for sel, ws, ts, tms, ln, cand, mt, lab in batches(ev_users, ev_rows, batch_users=128):
        p = torch.softmax(t3_model(*[t.to(device) for t in (ws, ts, tms, ln, cand, mt)]), dim=-1).cpu().numpy()
        for j, i in enumerate(sel):
            r = ev_pos[ev_rows[i]]; pj = p[j, :len(r)]
            maxprob[r] = pj.max(axis=1); pengaged[r] = 1.0 - pj[:, 0]
SC["score_t3_maxprob"], SC["score_t3_pengaged"] = maxprob, pengaged
SC["t3_list"] = K.t3_list.values[ev_mask]
print(f"scored {len(SC):,} rows with team 3's model")

scored 3,966,935 rows with team 3's model


**RESULT: team 3 vs the other baselines, on their lists and on full slates** (validation users)

Their lists keep every positive plus 4 negatives per positive (median 5 products), so almost any ranking puts a positive in the top 10. Full slates (~89 products) are the real task. The gap between the two columns is how much of their reported Hit@10 0.978 came from the protocol.

In [ ]:
T3 = ["score_t3_maxprob", "score_t3_pengaged"]
V = SC[SC.split.values == 1].reset_index(drop=True)
L = V[V.t3_list.values].reset_index(drop=True)
rows = []
for c in ["score_random", "score_pop", "score_repeat_pop"] + T3:
    lst, full = summarise(evaluate(L, L[c]), ""), summarise(evaluate(V, V[c]), "")
    rows.append({"model": BASELINE_NAMES.get(c, c), "their lists: hit@10": lst["hit@10"], "their lists: ndcg@10": lst["ndcg@10"],
                 "full slates: hit@10": full["hit@10"], "full slates: ndcg@10": full["ndcg@10"], "full slates: hit@10 95% CI": full["hit@10 95% CI"]})
proto = pd.DataFrame(rows).set_index("model")
display(proto.style.format({c: "{:.4f}" for c in proto.columns if "CI" not in c}))
proto.to_csv(f"{OUT_DIR}/step2_team3_protocol_check.csv")
SC = SC.drop(columns=["t3_list"])

,their lists: hit@10,their lists: ndcg@10,full slates: hit@10,full slates: ndcg@10,full slates: hit@10 95% CI
model,,,,,
Random,0.9928,0.5320,0.2240,0.0826,0.218 to 0.230
Popularity,0.9974,0.7491,0.6437,0.3448,0.637 to 0.651
Repeat → popularity,0.9976,0.7669,0.6813,0.3856,0.675 to 0.689
Team 3 NCF (as published: max class prob.),0.9858,0.5551,0.2533,0.1079,0.247 to 0.260
Team 3 NCF (fixed: P(engaged)),0.9931,0.6544,0.4507,0.2187,0.443 to 0.458


## 5. Save the scores for step 3

`baseline_scores.parquet` has one row per evaluation candidate: `user`, `wid`, `split`, `label`, `train_eval` and one `score_*` column per baseline. Step 3 evaluates **every** `score_*` column it finds, so adding the team 3 scores needs no change there.

In [ ]:
SC.to_parquet(f"{ART_DIR}/baseline_scores.parquet", index=False)
print("saved baseline_scores.parquet:", len(SC), "rows | score columns:", [c for c in SC.columns if c.startswith("score_")])
if IN_COLAB and UPLOAD_ARTIFACTS:
    drive_upload(STEP2_FILES)

saved baseline_scores.parquet: 3966935 rows | score columns: ['score_random', 'score_query', 'score_pop', 'score_repeat_pop', 'score_mf', 'score_t3_maxprob', 'score_t3_pengaged']


uploaded baseline_scores.parquet to My Drive/BT4222_artifacts  6s


## Findings (validation users, full slates)
- **Best baseline: repeat → popularity**, Hit@10 0.681, NDCG@10 0.386. Popularity 0.644 / 0.345; MF 0.603 / 0.320; query only 0.235 / 0.091, barely above random (0.224 / 0.083).
- **Cold products limit popularity and MF.** Popularity scores 0 for 30.3% of engaged rows and MF for 26.7%, so these tie with every other unseen product.
- **Team 3's reported score came from the protocol.** On their lists (every positive + 4 negatives), even random gets Hit@10 0.993. On full slates team 3's model gets 0.253 as published (max class probability) and 0.451 ranked by P(engaged), below popularity.
- Baseline scores on train users are in `step3_lightgbm` (Section 3).